코드 .master 함수를 사용하여 클러스터 매니저를 지정, local은 Spark를 로컬에서 실행한다는 의미이다. (1) 분산의 경우 master URL 또는 (2) 로컬인 경우 local[]라고 적어준다. 즉 local의 수는 CPU core의 수를 의미한다. 예를 들어 local[*]는 가능한 최대한의 core를 사용한다는 의미이다. 예를 들어, local[5]라고 하면, core의 수가 2개라고 하더라도 데이터는 5개의 partitions로 나누어져 주어진다.
local은 Spark를 로컬에서 실행한다는 의미이다.
local[n]는 worker의 쓰레드를 n개로 한다는 의미. CPU core의 개수에 맞추어 설정하자.
local[*] 는 가능하면 가용한 모든 쓰레드를 사용한다는 의미 (Runtime.getRuntime.availableProcessors()로 그 수를 알 수 있다)

In [5]:
%%writefile data/ds_spark_wiki.txt
Wikipedia
Apache Spark is an open source cluster computing framework.
아파치 스파크는 오픈 소스 클러스터 컴퓨팅 프레임워크이다.
Apache Spark Apache Spark Apache Spark Apache Spark
아파치 스파크 아파치 스파크 아파치 스파크 아파치 스파크
Originally developed at the University of California, Berkeley's AMPLab,
the Spark codebase was later donated to the Apache Software Foundation,
which has maintained it since.
Spark provides an interface for programming entire clusters with
implicit data parallelism and fault-tolerance.

Overwriting data/ds_spark_wiki.txt


In [6]:
RDD와 Spark Dataframe를 만드는 함수는 서로 다르다
DataFrame은 다음 장에서 배우게 되겠지만, file에서 읽는 방식이 RDD와 Dataframe이 서로 다르다. RDD는 sparkContext.textFile(), Dataframe은 read.text()을 사용한다.



myDf=spark.read.text(os.path.join("data", "ds_spark_wiki.txt"))

SyntaxError: invalid syntax (741958524.py, line 1)

In [7]:
%%writefile src/ds3_popCsvRead.py
#!/usr/bin/env python3
# -*- coding: UTF-8 -*-
import os
import pyspark

def doIt():
    print ("---------RESULT-----------")
    popDf = spark\
                .read.option("charset", "euc-kr")\
                .option("header", "true")\
                .csv(os.path.join("data","경기도 의정부시_인구현황_20240930.csv"))
    popDf.show(5)
    agedDf = spark\
                .read.option("charset", "euc-kr")\
                .option("header", "true")\
                .csv(os.path.join("data","제주특별자치도 서귀포시_고령화비율및노령화지수현황_20240419.csv"))
    agedDf.show(5)

if __name__ == "__main__":
    #os.environ["PYSPARK_PYTHON"]="/usr/bin/python3"
    #os.environ["PYSPARK_DRIVER_PYTHON"]="/usr/bin/python3"
    myConf=pyspark.SparkConf()
    spark = pyspark.sql.SparkSession.builder\
        .master("local")\
        .appName("myApp")\
        .config(conf=myConf)\
        .getOrCreate()
    doIt()
    spark.stop()

Overwriting src/ds3_popCsvRead.py


In [8]:
!spark-submit src/ds3_popCsvRead.py

Python


24/12/07 23:37:38 INFO ShutdownHookManager: Shutdown hook called
24/12/07 23:37:38 INFO ShutdownHookManager: Deleting directory C:\Users\rkdus3696\AppData\Local\Temp\spark-41da72b5-232f-44af-8f57-66de16f92588


In [9]:
#1. 변환 transformations
#-> 결과 실행시 나오는 결과가 RDD

In [10]:
#연산 식을 기억만 시켜놓은 것

In [11]:
#action 될때까지 연기하는 lazy이기 때문임

In [12]:
#2. actions
#-> 실행시 결과가 python 즉 list이다

In [13]:
#action때 실제 연산이 되는 것

In [14]:
#map, reduce함수가 사용 -> map reduce 알고리즘!

In [15]:
#이전의 프로그래밍과 이후가 다르다
#1부터 100까지 더한다고 하면 map reduce 이전에 반복문을 썼다 but 이후에는 반복문이 필요없다
#map reduce는 반복문 없고 numpy할 때 행렬 프로그램처럼 반복문이 없는 것
#1 - 100까지 합걔를 내는 게 완전 다르다

In [16]:
import os, sys
import pyspark


#아래의 설정은 jupyter notebook의 임시적으로 설정하는 방법이고
#영구적으로 아래와 같은 설정을 하기 위해서는 환경변수를 설정해줘야 한다


os.environ["PYSPARK_PYTHON"]=sys.executable          # Worker가 사용할 Python 실행 파일 경로 설정 (리눅스 "/usr/bin/python3")
os.environ["PYSPARK_DRIVER_PYTHON"]=sys.executable   # Driver에서도 동일한 Python 경로 설정
#os.environ['HADOOP_HOME']=os.getcwd() # 현재 디렉터리를 HADOOP_HOME으로 설정
#os.environ["PATH"] += os.path.join(os.environ['HADOOP_HOME'], 'bin') # PATH에 Hadoop 바이너리 추가

#driver랑 worker랑 python버전을 일치시키는 방법
#참고로 driver가 worker에게 지시를 내리기 때문에 두개의 버전을 일치시키는 것이 필요하다

myConf=pyspark.SparkConf() # 기본 설정 객체 생성, 여기에 필요한 설정 정의
#myConf=pyspark.SparkConf().set("spark.driver.bindAddress", "127.0.0.1") 드라이버 바인딩 주소 설정
#myConf=pyspark.SparkConf().set("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:10.1.1") 



###############SparkSessin 매우 중요##################

#spark를 시작할 땐 미리 아래의 코드를 생성해두는 것이 좋다

spark = pyspark.sql.SparkSession\
    .builder\
    .master("local")\
    .appName("myApp")\
    .config(conf=myConf)\
    .getOrCreate()


In [17]:
#map

nRdd = spark.sparkContext.parallelize([1, 2, 3, 4])
squared = nRdd.map(lambda x: x * x)

print (squared)

PythonRDD[1] at RDD at PythonRDD.scala:53


In [18]:
#x가 하나씩 꺼내진다
#map은 RDD함수 (map은 tranceforme)
#RDD는 출력 불가 action을 꼭 써야함 특별하게 저장되어있기 때문에 내부 확인 불가

In [19]:
myRdd4.take(5)

NameError: name 'myRdd4' is not defined

모든 줄을 반복:
    한 줄line을 읽는다. ('35, 2' 첫 반복에서 문자열)
    줄line을 컴마(,)로 분리한다. ('35', ' 2' 컴마로 분리했으므로 2앞에 공백이 없어지지 않고 남아있다)
    줄line을 리스트로 만든다. (['35', ' 2'] split()은 문자열을 분리해서 리스트로 만든다. 형변환하지 않으면 문자열이 유지된다)

In [20]:
myRdd5 = myRdd4.map(lambda line: line.split(','))
myRdd5.take(5)

NameError: name 'myRdd4' is not defined

In [ ]:
#data분석 :: data를 하나의 값으로 만드는 것
#키 data가 나열되어 있을 때 보통 평균키, 중앙값 등등의 원천데이터를 가지고 하나의 data로 합침(가공)

In [ ]:
#line 하나가 '35,2' 이렇게 한 묶음인 것

#35,2를 split 한다면? 
1) 문자열이므로 숫자가 될 순 없음
2) '35', '2' 라고 생각할 순 있지만 
3) 리스트 반환이라 ['35','2'] 이렇게 나뉜다

In [ ]:
x=['35', ' 2']
y=list()
for i in x:
    y.append(int(i))
print(y)

In [ ]:
#i는 리스트 ['35', ' 2']의 각 요소를 차례로 가리킵니다. 
#'35'를 꺼내서 int에 넣고 변환한 걸 리스트에 추가
[int(i) for i in ['35', ' 2']]
[str(i) for i in [35, 2]]

In [ ]:
myRdd6 = myRdd5.map(lambda x: [int(i) for i in x])
myRdd6.take(5)

In [ ]:
myRdd2=spark.sparkContext\
    .textFile(os.path.join("data","ds_spark_wiki.txt"))

In [ ]:
sentences=myRdd2.map(lambda x:x.split())

In [ ]:
def mySplit(x):
    return x.split()

sentences2=myRdd2.map(mySplit)
sentences2.count()

In [ ]:
#함수 이름에는 괄호가 있어야해 [함수] 라서 myRdd2에서 x가 하나씩 넘어가는 것
#람다 함수를 대신할 수 있다!
#람다랑 기능 차이는 없는데 람다는 이름이 없는 일회용 함수

In [ ]:
sentences.take(3)

In [ ]:
for line in sentences.collect():
    for word in line:
        print (word, end=" ") #여기서 단어 연결시 공백으로 연결하겠다는 뜻이니까 유용하게 쓰기!
    print ("\n-----")

In [ ]:
#문장이라서 단어를 볼라면 중첩 반복문이 필요다다

In [ ]:
len("Apache Spark is an open source cluster computing framework")

In [ ]:
myRdd2.map(lambda s:len(s)).collect()

In [ ]:
#각 인덱스 안에 문장이 들어가 있다는 걸 알 수 있다

In [ ]:
#원래는 마침표가 들어있어서 하나 플러스가 된다

In [ ]:
myList=["this is","a line"]
_rdd=spark.sparkContext.parallelize(myList)

In [ ]:
repRdd=_rdd.map(lambda x:x.replace("this","This"))
repRdd.take(10)

In [ ]:
's'.upper()

In [ ]:
wordsRdd=_rdd.map(lambda x:x.split())
print (wordsRdd.collect())

In [ ]:
upperRDD =wordsRdd.map(lambda x: x[0].upper())
print (upperRDD.collect())

In [ ]:
upper2RDD =wordsRdd.map(lambda x: [i.upper() for i in x])
print (upper2RDD.collect())

In [ ]:
myRdd100 = spark.sparkContext.parallelize(range(1,101))

In [ ]:
#map 는 변환 함수!
#reduce action은 값

In [ ]:
myRdd100.reduce(lambda subtotal, x: subtotal + x)

In [ ]:
#parallelize

In [ ]:
spark.sparkContext.parallelize(range(1,11),2).glom().collect()

In [ ]:
spark.sparkContext.parallelize(range(1,11)).fold(0, lambda subtotal, x: subtotal + x)

In [ ]:
#range는 뒤에 값이 exclusive이다

In [ ]:
spark.sparkContext.parallelize(range(1,11),1).fold(10, lambda subtotal, x: subtotal + x)

In [ ]:
#reduce는 인자가 항상 2개야 
#1. 토탈 2. 결과값을 넣을 곳

In [ ]:
#반복문 없이 map reduce로 할 수 있따
myRdd100.reduce(lambda subtotal, x: subtotal + x)

In [ ]:
spark.sparkContext.parallelize(range(1,11),2).glom().collect()

In [ ]:
spark.sparkContext.parallelize(range(1,11)).fold(0, lambda subtotal, x: subtotal + x)

In [ ]:
spark.sparkContext.parallelize(range(1,11),1).fold(10, lambda subtotal, x: subtotal + x)

In [ ]:
spark.sparkContext.parallelize(range(1,11),2).fold(10, lambda subtotal, x: subtotal + x)

In [ ]:
#fold는 리듀스랑 거의 비슷한데 초기값을 지정한 값으로 계속 더해준다는 것이 특징

In [ ]:
#넘파이는 함수를 이용하여 바로 쓸 수 있다 (배열프로그래밍 이기 때문)
print ("sum: ", myRdd100.sum())
print ("min: ", myRdd100.min())
print ("max: ", myRdd100.max())
print ("count: ", myRdd100.count())
print ("standard deviation:", myRdd100.stdev())
print ("variance: ", myRdd100.variance())

In [ ]:
myRdd_spark=myRdd2.filter(lambda line: "Spark" in line)
print ("How many lines having 'Spark': ", myRdd_spark.count())
#filter은 분리!

 stopwords  불용어
 = 제거해야하는 단어

한 글자짜리 단어들이 있는데 주제를 형

pipeline::
결과를 받아서 다음으로 연결해주는 방식이다
입출력이 연결된다

In [ ]:
upper2list=wordsRdd.map(lambda x: [i.upper() for i in x]).collect()
print (upper2list)

In [ ]:
wordsLength = wordsRdd\
    .map(len)\
    .collect()
print (wordsLength)

In [17]:
myRdd2.take(10)

NameError: name 'myRdd2' is not defined

In [18]:
#myRdd_group=myRdd2.groupBy(lambda x:x[0:2])
myRdd_group=myRdd2.groupBy(lambda x:"아파치" in x)

for (k,v) in myRdd_group.collect():
    print ("{}: {}".format(k, v))

NameError: name 'myRdd2' is not defined

In [19]:
#myRdd_group=myRdd2.flatMap(lambda x:x.split()).groupBy(lambda x:w[0:2])
#myRdd_group=myRdd2.groupBy(lambda x:x[0:2])

for (k,v) in myRdd_group.collect():
    for eachValue in v:
        print ("{}: {}".format(k, eachValue))
    print ("-----")

NameError: name 'myRdd_group' is not defined